# Phase 2 — Baseline CNN
Run this notebook on Kaggle. After training, all results are saved to `results/`.
Download `results/metrics.json` and the comparison images and hand them to Claude.

In [ ]:
!pip install -q torchmetrics scikit-image

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torchmetrics.image import StructuralSimilarityIndexMeasure as SSIM
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric

os.makedirs('results', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
DATA_ROOT    = '/kaggle/input/datasets/nachiketpatil0105/spc-data/train/'
NUM_FRAMES   = 128
LEARNING_RATE = 1e-4
NUM_EPOCHS   = 50

In [ ]:
path = Path(DATA_ROOT)
npy_files = sorted(path.rglob('*.npy'))
png_files = sorted(path.rglob('*.png'))

train_X, train_y = npy_files[:45], png_files[:45]
test_X,  test_y  = npy_files[-5:], png_files[-5:]

print(f'Train: {len(train_X)}  Test: {len(test_X)}')

In [ ]:
def load_input(npy_path):
    data = np.load(npy_path, mmap_mode='r')[-NUM_FRAMES:]
    data = np.unpackbits(data, axis=2)
    data = data.reshape(NUM_FRAMES, 800, 800, 3)
    data = np.transpose(data, (1, 2, 0, 3))
    data = data.reshape(800, 800, NUM_FRAMES * 3)
    data = data / data.max()
    return data.transpose(2, 0, 1).astype(np.float32)

def load_target(png_path):
    img = np.array(Image.open(png_path)).astype(np.float32) / 255.0
    return img.transpose(2, 0, 1)

def to_tensor(arr):
    return torch.from_numpy(arr).float().unsqueeze(0).to(device)

In [ ]:
class SimCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(384, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(),
            nn.Conv2d(256, 128, 3, padding=1), nn.ReLU(),
            nn.Conv2d(128,  64, 3, padding=1), nn.ReLU(),
            nn.Conv2d( 64,  32, 3, padding=1), nn.ReLU(),
            nn.Conv2d( 32,  16, 3, padding=1), nn.ReLU(),
            nn.Conv2d( 16,   3, 3, padding=1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
model     = SimCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
ssim_fn   = SSIM(data_range=1.0).to(device)
l1_fn     = nn.L1Loss()

epoch_losses = []
model.train()

for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    for npy_path, png_path in zip(train_X, train_y):
        x      = to_tensor(load_input(npy_path))
        target = to_tensor(load_target(png_path))

        pred = model(x)
        loss = 0.5 * (1 - ssim_fn(pred, target)) + 0.5 * l1_fn(pred, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg = total_loss / len(train_X)
    epoch_losses.append(avg)
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}  Loss: {avg:.4f}')

In [ ]:
# Loss curve
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, NUM_EPOCHS+1), epoch_losses, marker='o', markersize=3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss Curve', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/loss_curve.png')

In [ ]:
model.eval()
test_results = []

with torch.no_grad():
    for i, (npy_path, png_path) in enumerate(zip(test_X, test_y)):
        x      = to_tensor(load_input(npy_path))
        target = to_tensor(load_target(png_path))
        pred   = model(x)

        pred_np   = np.clip(pred.squeeze().cpu().numpy().transpose(1,2,0), 0, 1)
        target_np = target.squeeze().cpu().numpy().transpose(1,2,0)
        input_np  = x.squeeze().cpu().numpy()[:3].transpose(1,2,0)

        p = psnr_metric(target_np, pred_np, data_range=1.0)
        s = ssim_metric(target_np, pred_np, channel_axis=2, data_range=1.0)

        scene = Path(png_path).parent.name
        test_results.append({'sample': i, 'scene': scene, 'PSNR': float(p), 'SSIM': float(s)})
        print(f'Sample {i} ({scene})  PSNR={p:.2f} dB  SSIM={s:.4f}')

        # Save comparison figure
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle(f'CNN Reconstruction — {scene}', fontsize=13, fontweight='bold')
        for ax, (img, title, sub) in zip(axes, [
            (input_np,  'SPC Input (first 3 ch)', ''),
            (pred_np,   'CNN Prediction',          f'PSNR={p:.2f} dB  SSIM={s:.4f}'),
            (target_np, 'Ground Truth',            ''),
        ]):
            ax.imshow(np.clip(img, 0, 1))
            ax.set_title(title, fontsize=11, fontweight='bold')
            ax.set_xlabel(sub, fontsize=9)
            ax.axis('off')
        plt.tight_layout()
        plt.savefig(f'results/comparison_sample{i}_{scene}.png', dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()

avg_psnr = sum(r['PSNR'] for r in test_results) / len(test_results)
avg_ssim = sum(r['SSIM'] for r in test_results) / len(test_results)
print(f'\nAverage  PSNR={avg_psnr:.2f} dB  SSIM={avg_ssim:.4f}')

In [ ]:
with open('results/metrics.json', 'w') as f:
    json.dump({
        'test_results': test_results,
        'epoch_losses': epoch_losses,
        'avg_psnr': avg_psnr,
        'avg_ssim': avg_ssim,
    }, f, indent=2)
print('Saved → results/metrics.json')
print('\nDownload the entire results/ folder and share with Claude.')